<a href="https://colab.research.google.com/github/laurianedlz/Fundamentos-de-Optimizacion/blob/main/Practico_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradiente condicional y gradiente proyectado

Consideremos el problema:
$$x^* = \arg \min_{x \in \mathcal{X}} f(x)$$

donde $f(x) = \frac{1}{2} (x_1^2 + x_2^2 + 0,1 x_3^2) + 0,55 x_3$, y

$$\mathcal{X} = \{x = (x_1, x_2, x_3) \in \mathbb{R}^n : x_1 + x_2 + x_3 = 1, 0 \le x_1, 0 \le x_2, 0 \le x_3\}$$

Se sabe que el mínimo global es $x^* = (\frac{1}{2}, \frac{1}{2}, 0)$.

a) Implementar una función que, dado $x \in \mathbb{R}^3$, devuelva $P_{\mathcal{X}}(x)$ para el conjunto $\mathcal{X}$ de este problema.
La proyección de $y \in \mathbb{R}^3$ sobre $\mathcal{X}$ tiene la forma:
$$P_{\mathcal{X}}(y)_i = \max(y_i - \mu, 0), \quad i = 1, 2, 3$$
donde $\mu \in \mathbb{R}$ es la única constante tal que $\sum_{i=1}^{3} \max(y_i - \mu, 0) = 1$.

In [1]:
import numpy as np

def projection(x):
  y=sorted(x, reverse=True)
  sum_1=y[0]
  sum_2=y[0]+y[1]
  sum_3=y[0]+y[1]+y[2]
  k=1
  if(y[1]+(1-sum_2)/2>0):
    k=2
    if(y[2]+(1-sum_3)/3>0):
      k=3
  if k==1:
    mu=(sum_1-1)/1
  elif k==2:
    mu=(sum_2-1)/2
  else:
    mu=(sum_3-1)/3
  proy_1=max(x[0]-mu,0)
  proy_2=max(x[1]-mu,0)
  proy_3=max(x[2]-mu,0)
  return [proy_1,proy_2,proy_3]

b) Implementar el método de Frank-Wolfe con regla de paso dada por minimización en la dirección (line search). Notar que, en este caso, existe una expresion en forma cerrada para el paso optimo.

Verificar computacionalmente que, para cualquier punto inicial $(\xi_1, \xi_2, \xi_3)$ con $\xi_i > 0$ para todo $i$, y $\xi_1 \neq \xi_2$, la tasa de convergencia no es lineal en el sentido de que:
$$\frac{f(x^{k+1}) - f(x^*)}{f(x^k) - f(x^*)} \to 1$$

In [31]:
def gradf (x):
  return np.array([x[0],x[1],0.1*x[2]+0.55])

def f (x):
  return 0.5*(x[0]**2+x[1]**2+0.1*x[2]**2)+0.55*x[2]

x_opt=np.array([0.5,0.5,0])
Q = np.diag([1, 1, 0.1])

def frank_wolfe_proximo_x (gradf,x):
  if min(gradf(x)) == gradf(x)[0]:
    s=np.array([1,0,0])
  elif min(gradf(x)) == gradf(x)[1]:
    s=np.array([0,1,0])
  else:
    s=np.array([0,0,1])
  d=s-x
  ##Tenemos que buscar el minimo entre x y s
  ##Vamos a minimizar la function g(p)=x+pd con una formula que explico abajo
  p=-(gradf(x)@d)/((Q@d)@d)
  if p>1 :
    p=1
  elif p<0 :
    p=0
  return x+p*d

def frank_wolfe (x0,n,gradf):
  x=x0
  for i in range(n):
    old_x=x
    x=frank_wolfe_proximo_x(gradf,x)
  return np.array([old_x,x])

def tasa_convergencia(x,f):
  t=(f(x[1])-f(x_opt))/(f(x[0])-f(x_opt))
  return t

x1=np.array([1,0.5,0])
x2=np.array([0.5,0,0])
x3=np.array([0,0.5,0.5])

print(frank_wolfe(x1,10,gradf))
print(frank_wolfe(x2,10,gradf))
print(frank_wolfe(x3,10,gradf))

print(tasa_convergencia(frank_wolfe(x1,10,gradf),f))
print(tasa_convergencia(frank_wolfe(x1,11,gradf),f))
print(tasa_convergencia(frank_wolfe(x1,12,gradf),f))

print(tasa_convergencia(frank_wolfe(x2,10,gradf),f))
print(tasa_convergencia(frank_wolfe(x2,11,gradf),f))
print(tasa_convergencia(frank_wolfe(x2,12,gradf),f))

print(tasa_convergencia(frank_wolfe(x3,10,gradf),f))
print(tasa_convergencia(frank_wolfe(x3,11,gradf),f))
print(tasa_convergencia(frank_wolfe(x3,12,gradf),f))



[[0.49818585 0.54255416 0.        ]
 [0.53894997 0.49848059 0.        ]]
[[0.4 0.2 0. ]
 [0.4 0.2 0. ]]
[[0.45639827 0.42799139 0.11561034]
 [0.4322475  0.4582598  0.1094927 ]]
0.9153034085347662
0.9224189607042672
0.9283960087470695
1.0
1.0
1.0
0.9247807403191325
0.931377455563084
0.9369055152159209


Queremos minimizar este funcion : $$g(p) = f(x + p d)$$

Como $f$ es cuadratica, podemos buscar el punto donde : $g'(p) = 0$.

$$g'(p) = \langle \nabla f(x + p d), d \rangle$$

Tenemos $\nabla f(x) = Qx + c$, où :

$$Q = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0,1 \end{pmatrix}, \quad c = \begin{pmatrix} 0 \\ 0 \\ 0,55 \end{pmatrix}$$

Entonces cuando developamos (dejamos el $c$ en el $\nabla f(x)$), tenemos:

$$g'(p) = \langle \nabla f(x) + p Q d, d \rangle = \langle \nabla f(x), d \rangle + p \langle Q d, d \rangle$$

Hacemos $g'(p) = 0$, y encontremos la formula para el $p$ que va a minimizar g :

$$p = \frac{-\langle \nabla f(x), d \rangle}{\langle Q d, d \rangle}$$



c) Consideramos ahora el método de gradiente proyectado aplicado al problema anterior, es decir:
$$x^{k+1} = x^k + \alpha^k d^k, \quad d^k = \bar{x}^k - x^k, \quad \bar{x}^k = P_{\mathcal{X}}(x^k - s^k \nabla f(x^k))$$

En este ejercicio, tomamos $\alpha^k = 1$, de modo que:
$$x^{k+1} = P_{\mathcal{X}}(x^k - s^k \nabla f(x^k))$$

Implementar el método utilizando la misma regla de paso de la parte anterior.

In [30]:
def gradf (x):
  return np.array([x[0],x[1],0.1*x[2]+0.55])

def f (x):
  return 0.5*(x[0]**2+x[1]**2+0.1*x[2]**2)+0.55*x[2]

x_opt=np.array([0.5,0.5,0])
Q = np.diag([1, 1, 0.1])

def grad_proy_proximo_x (gradf,x):
  d=-gradf(x)
  p=-(gradf(x)@d)/((Q@d)@d) ##como antes
  return projection(x-p*gradf(x))

def grad_proy (x0,n,gradf):
  x=x0
  for i in range(n):
    old_x=x
    x=grad_proy_proximo_x(gradf,x)
  return np.array([old_x,x])

def tasa_convergencia(x,f):
  t=(f(x[1])-f(x_opt))/(f(x[0])-f(x_opt))
  return t

x1=np.array([1,0.5,0])
x2=np.array([0.5,0,0])
x3=np.array([0,0.5,0.5])

print(grad_proy(x1,10,gradf))
print(grad_proy(x2,10,gradf))
print(grad_proy(x3,10,gradf))

print(tasa_convergencia(grad_proy(x1,10,gradf),f))
print(tasa_convergencia(grad_proy(x1,11,gradf),f))
print(tasa_convergencia(grad_proy(x1,12,gradf),f))

print(tasa_convergencia(grad_proy(x2,10,gradf),f))
print(tasa_convergencia(grad_proy(x2,11,gradf),f))
print(tasa_convergencia(grad_proy(x2,12,gradf),f))

print(tasa_convergencia(grad_proy(x3,10,gradf),f))
print(tasa_convergencia(grad_proy(x3,11,gradf),f))
print(tasa_convergencia(grad_proy(x3,12,gradf),f))


[[0.4997469  0.5002531  0.        ]
 [0.50012995 0.49987005 0.        ]]
[[0.4990886  0.5009114  0.        ]
 [0.50046794 0.49953206 0.        ]]
[[0.50099148 0.49900852 0.        ]
 [0.49949094 0.50050906 0.        ]]
0.26361748465485063
0.26361757888408144
0.2636176017955412
0.26361596065497905
0.263617177150877
0.2636174980103102
0.26361565762563793
0.2636170970953681
0.26361747648997097


d) Comparar numéricamente la convergencia de los dos métodos implementados: de gradiente proyectado con la del método de gradiente condicional considerado en el item anterior.

Podemos ver que cuando hacemos 10, 11, 12 iteraciones, el método del gradiente proyectado (gp) tiene un tasa de convergencia muy pequeno y quasi constante, comparado al del gradiente condicional (gc) que tiende a 1. Significa que al paso 10, el método de gp continua majorar sur resultado cuando el gc no lo hace mucho.
Para este problema, el gp es mas eficiente, mientras que gc requiere mucho mas iteraciones para lograr una precision alta (debido a su ratio cercano a 1), el gp alcanza el optimo en muy pocos pasos gracias a su racio pequeno.